## Instructions for this notebook:

Prior to running this notebook, the following data must be downloaded and installed in the project folder under Data/Raw/manning/:

[CLRH Hydrofabric dataset](https://hydrology.uwaterloo.ca/CLRH/Hydrofabric.html) (download watershed of interest based on outlet point) 
- Extract zip file to: /Data/Raw/Manning/CLRH_RoutingProduct/\[station ID\]

[ACI raster](https://open.canada.ca/data/en/dataset/ba2645d5-4458-414d-b196-6303ac06c1c9) (pre-clipped to watershed of interest in ArcGIS Pro)
- Extract zip file to: /Data/Raw/Manning/ACI_2023

crop_categories_data.csv (CSV containing crop cover period and P addition/uptake estimates for each crop type)
- Save to: /Data/Raw/Manning/ACI_2023

## Process ACI Raster

NOTE: ACI raster was pre-clipped to manning canal watershed in ArcGIS Pro

In [1]:
import os
import sys
import fileinput
import numpy as np
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, LineString, Polygon, MultiPoint
#from datetime import datetime
import rasterio
import rasterio.mask
import fiona

sys.path.append("..")
from src.py.proj_conf import *

Warning: changing project projection for simplicity (no pre-built functions exist for easy reprojection of GeoTIFF files)

In [2]:
# Project mc subbasins to align with aci raster
with rasterio.open("/Users/emilito/Documents/PWA/Data/Raw/manning/ACI_2023/aci_2023_clip_mc_clrh.tif") as aci_src:
    print(aci_src.crs)

mc_subbasins = gpd.read_file("/Users/emilito/Documents/PWA/Data/Processed/manning/Subbasins/mc_clrh_subbasins.shp")
mc_subbasins_project = mc_subbasins.to_crs(3857)

EPSG:3857


In [3]:
# Create list of subbasin IDs
subids = list(mc_subbasins_project['SubId'])

In [4]:
# Define land cover classes
CLASS_NAMES = {
    1: "Winter wheat",
    2: "Spring wheat",
    4: "Barley",
    5: "Millet",
    6: "Oats",
    7: "Rye",
    16: "Other grains",
    19: "Canola/rapeseed",
    23: "Sunflower",
    24: "Soybeans",
    25: "Hemp",
    26: "Peas",
    28: "Beans",
    32: "Corn"
}

In [5]:
# Calculate percent of each land cover type per subbasin

# Open the raster file
with rasterio.open("/Users/emilito/Documents/PWA/Data/Raw/manning/ACI_2023/aci_2023_clip_mc_clrh.tif") as src:
    results = []

    # Iterate over each subbasins polygon
    for index, row in mc_subbasins_project.iterrows():
        polygon = [row["geometry"]]
        
        # Mask (crop) the raster to the polygon
        out_image, out_transform = rasterio.mask.mask(src, polygon, crop=True)

        # Remove NoData values (0 & 255) and non-crop land cover types
        valid_data = out_image[np.isin(out_image, [0, 45, 46, 48, 55, 57, 58, 59, 60, 61, 62, 63, 64, 255], invert=True)].flatten()

        if len(valid_data) == 0:
           continue  # Skip empty polygons

        # Find unique classes and their counts
        unique_classes, counts = np.unique(valid_data, return_counts=True)

        # Compute total valid pixels
        total_pixels = counts.sum()

        # Compute percentages
        class_percentages = {cls: (count / total_pixels) * 100 for cls, count in zip(unique_classes, counts)}

        # Store results
        results.append({"Polygon_ID": subids[index], "Class_Percentages": class_percentages})

# Output results as csv
data = []
for res in results:
    subbasin_id = res["Polygon_ID"]  # Treat as column names

    # Convert NumPy types to Python native types
    class_percentages = {int(cls): float(percent) for cls, percent in res["Class_Percentages"].items()}
    
    for cls, percent in class_percentages.items():
        crop_type = CLASS_NAMES.get(cls, f"Class {cls}")  # Use name or default to "Class X"
        data.append([crop_type, subbasin_id, percent])

# Convert to DataFrame
df = pd.DataFrame(data, columns=["Crop_Type", "Subbasin_ID", "Percentage"])

# Pivot the table to have Crop_Type as rows and Subbasin_ID as columns
df_pivot = df.pivot(index="Crop_Type", columns="Subbasin_ID", values="Percentage").fillna(0)  # Replace NaN with 0

if not os.path.exists("/Users/emilito/Documents/PWA/Data/Processed/manning/ACI"):
    os.makedirs("/Users/emilito/Documents/PWA/Data/Processed/manning/ACI")

# Save the pivoted DataFrame to CSV
df_pivot.to_csv("/Users/emilito/Documents/PWA/Data/Processed/manning/ACI/percent_crop_type_per_subbasin.csv")

print("CSV saved successfully")

CSV saved successfully


## Calculate LULC Features per Subbasin

NOTE: THE FOLLOWING CODE WAS DERIVED FROM R CODE CREATED FOR PILOT 1 (feature_creation.R) AND WAS MODIFIED FOR PILOT 2

NEEDS TROUBLESHOOTING

In [6]:
### CROP CATEGORIES DATA IS INCOMPLETE FOR MANNING CANAL - SEE ISSUE LOG

# Import crop data
crop_data = pd.read_csv("/Users/emilito/Documents/PWA/Data/Raw/manning/ACI_2023/crop_categories_data.csv")
prcnt_crop = pd.read_csv("/Users/emilito/Documents/PWA/Data/Processed/manning/ACI/percent_crop_type_per_subbasin.csv")

In [7]:
# Adjust date values to days in a single calendar year
if crop_data['start_date'].mean() > 44000:
    crop_data['start_date'] = crop_data['start_date'] - 44926

    crop_data['end_date'] = np.where(
        (crop_data['end_date'] - 45292) < crop_data['start_date'],
        crop_data['end_date'] - 44926,
        crop_data['end_date'] - 45292
    )

In [8]:
# Calculate max and min fetilization weighted average per subbasin
max_f_wa = []
for column_name in prcnt_crop:
    if column_name != "Crop_Type":
        max_f_wa.append(np.average((crop_data["max_add"] - crop_data["min_uptk"]), weights=prcnt_crop[column_name]))

min_f_wa = []
for column_name in prcnt_crop:
    if column_name != "Crop_Type":
        min_f_wa.append(np.average((crop_data["min_add"] - crop_data["max_uptk"]), weights=prcnt_crop[column_name]))

# Create list of subbasins that contain cropland
subids_corr = []
for column_name in prcnt_crop:
    if column_name != "Crop_Type":
        subids_corr.append(column_name)

# Create lulc features output dataframe
d = {'SubId': subids_corr, 'max_f_wa': max_f_wa, 'min_f_wa': min_f_wa}
wq_crop_outputs = pd.DataFrame(data=d)
wq_crop_outputs

,SubId,max_f_wa,min_f_wa
0,310013151.0,-1.142857,-22.428571
1,310013152.0,0.000000,-19.000000
2,310013155.0,-5.000000,-28.000000
3,310013156.0,0.026338,-19.852034
4,310013183.0,1.169567,-19.104418
5,310013184.0,-1.879397,-23.492462
6,310015221.0,5.373093,-12.563288
7,310015222.0,3.595580,-14.691161
8,310015223.0,4.541786,-13.532153
9,310015224.0,6.519523,-10.993116


In [ ]:
# Calculate crop cover period (weighted start date and weighted end date)

# Convert code from feature_creation.R and modify for manning canal

In [ ]:
# Create crop cover timeseries (calculate hours since seeding and harvest for each year)

# Convert code from feature_creation.R and modify for manning canal